# 02 · Construcción del dataset de modelado

**Objetivo:** construir `dataset_modelo.parquet` (una fila = una cita) aplicando las reglas de la auditoría (notebook 01) y calculando todas las variables.

**¿Por qué las funciones están en `funciones_inasistencia.py` y no aquí?**
Porque la herramienta diaria (`prediccion_diaria.py`) tiene que calcular las variables de las citas de mañana **exactamente igual** que en el entrenamiento. Si copiáramos el código en dos lugares, tarde o temprano se desincronizarían, y el modelo recibiría en producción variables distintas a las que aprendió. A esto se le llama *training-serving skew*. Aquí llamamos las funciones paso a paso y revisamos el resultado de cada una.

### Reglas vigentes
| Regla | Motivo |
|---|---|
| Ventana 02-ene a 11-sep-2026 | RIPS no tiene rezago (notebook 01, sección 1.4) |
| Solo citas asignadas al menos 1 día antes | El modelo predice en D-1 |
| Solo consultas (sin procedimientos) | El RIPS disponible es de consultas |
| Sin terapias, sin especialidades no reportadas ni con registro inestable | Ruido de registro |
| Sin días sin reporte en RIPS (z robusto < −4) | Falla de reporte, no inasistencia |
| Sin posibles reprogramaciones | No son inasistencias |
| Servicio homologado | La agenda cambió su codificación en abril (notebook 01, sección 1.6) |
| Sin citas huérfanas de códigos descontinuados | Tras el cambio de abril, las citas con código viejo casi nunca aparecen en RIPS |
| Toda variable usa información hasta D-2 | En D-1 aún no se sabe qué pasó ese día |

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import funciones_inasistencia as fi
import funciones_modelado as fm

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
fm.estilo_graficos()

import json
from pathlib import Path

Path("modelos").mkdir(exist_ok=True)
params = dict(fi.PARAMETROS)
params

{'FECHA_INICIO': Timestamp('2026-01-02 00:00:00'),
 'FECHA_FIN_DATOS': Timestamp('2026-09-11 00:00:00'),
 'FIN_TRAIN': Timestamp('2026-05-31 00:00:00'),
 'FIN_VALIDACION': Timestamp('2026-06-30 00:00:00'),
 'FIN_TEST': Timestamp('2026-07-31 00:00:00'),
 'TERAPIAS_EXCLUIDAS': ['FISIOTERAPIA',
  'FONOAUDIOLOGIA',
  'TERAPISTA RESPIRATORIO',
  'TERAPIA OCUPACIONAL'],
 'SERVICIOS_REGISTRO_INESTABLE': ['AUXILIAR EN SALUD ORAL'],
 'UMBRAL_REPORTE_RIPS': 0.25,
 'MIN_CITAS_PARA_EVALUAR': 20,
 'UMBRAL_Z_DIA_SIN_REPORTE': -4.0,
 'MIN_CITAS_DIA_REGION': 100,
 'FECHA_CAMBIO_CODIFICACION': Timestamp('2026-04-01 00:00:00'),
 'CAIDA_MINIMA_DESCONTINUADO': 0.9,
 'MIN_CITAS_ANTES_CAMBIO': 300,
 'REZAGO_DIAS': 1,
 'VENTANA_MEDICAMENTOS': 90,
 'UMBRAL_POLIFARMACIA': 5,
 'VENTANA_ATENCIONES': 90,
 'DURACION_GESTACION': 280,
 'K_SUAVIZADO': 3}

## 1. Carga y limpieza de agendamiento

In [2]:
fuentes = fi.cargar_fuentes(".")
bitacora = []
rips = fi.preparar_rips(fuentes["rips"])
agenda = fi.limpiar_agendamiento(fuentes["agendamiento"], bitacora)

print("\nHoras sin convertir:", agenda["minuto_del_dia"].isna().sum())
agenda["tipo_agenda"].value_counts()

Agendamiento original: 576,252 filas
Sin duplicados exactos: 574,730 filas


Con fecha de asignación, especialidad y hora: 567,212 filas


Una cita por paciente-día-servicio: 516,498 filas



Horas sin convertir: 0


tipo_agenda
Consulta         468045
Procedimiento     48453
Name: count, dtype: int64

## 2. Variable objetivo con todas las reglas de la auditoría
`construir_citas_etiquetadas` devuelve:
- `etiquetadas`: las citas válidas para el modelo (con `no_asiste`).
- `todas`: todas las citas limpias con su marca de asistencia y de reprogramación (se usan para las variables v2).
- `mapa_servicio` y `excluidas`: lo aprendido, que se guarda para reutilizarlo en la predicción diaria.

In [3]:
etiquetadas, todas, mapa_servicio, excluidas = fi.construir_citas_etiquetadas(agenda, rips, params, bitacora=bitacora)
print("\nEspecialidades excluidas:", excluidas)
print("Códigos de agenda descontinuados:", etiquetadas.attrs.get("descontinuados"))

Dentro de la ventana con RIPS: 483,541 filas


Asignadas al menos 1 día antes (lead time >= 1): 442,516 filas
Solo consultas (sin procedimientos): 399,008 filas
Sin terapias ni especialidades no reportadas: 381,955 filas


Sin citas huérfanas de códigos descontinuados: 378,598 filas
Sin días sin reporte en RIPS: 359,677 filas
Sin posibles reprogramaciones: 337,375 filas



Especialidades excluidas: ['AUXILIAR EN SALUD ORAL', 'FISIOTERAPIA', 'FONOAUDIOLOGIA', 'HIGIENE ORAL', 'TERAPIA OCUPACIONAL', 'TERAPISTA RESPIRATORIO']
Códigos de agenda descontinuados: ['ADOLESCENCIA - ENF', 'ADULTEZ - MD GRAL', 'CIRUGIA MAXILOFACIAL', 'CONTROL PRENATAL - MEDICO', 'GINECOLOGIA', 'HTA/CRONICO - MEDICO', 'INFANCIA - ENF', 'PLANIFICACION FAMILIAR - ENFERMERIA', 'PRIMERA INFANCIA - ENF', 'PRIMERA INFANCIA - MD GRAL', 'SALUD MENTAL - MD GRAL', 'SALUD PUBLICA - MEDICO', 'VEJEZ - MD GRAL']


In [4]:
etiquetadas["cita_id"] = np.arange(len(etiquetadas))
etiquetadas["particion"] = etiquetadas["fecha_cita"].apply(lambda f: fi.asignar_particion(f, params))

# Prior de la tasa suavizada: SOLO con el periodo de entrenamiento (evita fuga de información)
params["PRIOR_INASISTENCIA"] = float(etiquetadas.loc[etiquetadas["particion"] == "train", "no_asiste"].mean())
print(f"Prior (tasa de inasistencia en train): {params['PRIOR_INASISTENCIA']:.3f}")

resumen = etiquetadas.groupby("particion")["no_asiste"].agg(citas="size", tasa_inasistencia="mean")
resumen["desde"] = etiquetadas.groupby("particion")["fecha_cita"].min().dt.date
resumen["hasta"] = etiquetadas.groupby("particion")["fecha_cita"].max().dt.date
resumen.loc[["train", "validacion", "test", "produccion_simulada"]].round(3)

Prior (tasa de inasistencia en train): 0.320


,citas,tasa_inasistencia,desde,hasta
particion,,,,
train,211402,0.320,2026-01-02,2026-05-30
validacion,38719,0.335,2026-06-01,2026-06-30
test,41148,0.312,2026-07-01,2026-07-31
produccion_simulada,46106,0.306,2026-08-01,2026-09-11


## 3. Variables
`calcular_variables` aplica en orden:

| Función | Variables |
|---|---|
| `variables_agendamiento` | hora (cíclica y franja), control, lead time, familia de servicio, PAC, edad y curso de vida |
| `variables_calendario` | día de la semana, festivos, puentes, inicio/fin de mes |
| `variables_historial` | citas previas, inasistencias previas, tasa suavizada, estado de la última cita (hasta D-2) |
| `variables_historial(por paciente y especialidad)` | lo mismo, pero solo en la especialidad de la cita (v2) |
| `variables_clinicas` | RCV, gestante, medicamentos 90 días, diagnósticos previos, atenciones 90 días (hasta D-2) |
| `variables_reprogramaciones_previas` | cuántas veces reprogramó/canceló antes (v2) |
| `variables_carga_agenda` | otras citas el mismo día y en ±7 días, que ya estaban agendadas en D-1 (v2) |
| `variables_interacciones` | familia × lead time, curso de vida × control (v2) |

### ¿Cómo se garantiza "solo información hasta D-2" en el historial?
Con `pd.merge_asof`: para cada cita buscamos **el último acumulado conocido** del paciente con fecha ≤ D-2. Es como preguntar: *"¿cuántas citas y fallas llevaba este paciente dos días antes de su cita?"*.

In [5]:
dataset = fi.calcular_variables(etiquetadas, etiquetadas, todas, agenda, fuentes, rips, params)
print(dataset.shape)

(337375, 77)


### Verificación anti-leakage del historial
Tomamos un paciente con varias citas y comprobamos a mano que el historial solo cuenta citas de **al menos 2 días antes**.

In [6]:
paciente = dataset["id_paciente"].value_counts().index[50]
columnas_revision = ["fecha_cita", "especialidad", "no_asiste", "n_citas_previas", "n_inasistidas_previas",
                     "tasa_inasistencia_suavizada", "estado_ultima_cita", "dias_desde_ultima_cita"]
dataset[dataset["id_paciente"] == paciente].sort_values("fecha_cita")[columnas_revision].head(12)

,fecha_cita,especialidad,no_asiste,n_citas_previas,n_inasistidas_previas,tasa_inasistencia_suavizada,estado_ultima_cita,dias_desde_ultima_cita
63424,2026-01-19,CONTROL PRENATAL - MEDICO,1,0,0,0.320276,Sin historial,NaN
63425,2026-01-22,OBSTETRICIA - GINECOLOGO,0,1,1,0.490207,No asistió,3.0
63426,2026-01-26,CONTROL PRENATAL - MEDICO,1,2,1,0.392166,Asistió,4.0
63427,2026-01-27,CRONICO - PSICOLOGIA,0,2,1,0.392166,Asistió,5.0
63428,2026-01-27,PSICOLOGIA,0,2,1,0.392166,Asistió,5.0
63429,2026-01-31,CRONICO - NUTRICIONISTA,1,5,2,0.370104,Asistió,4.0
63430,2026-01-31,NUTRICIONISTA,1,5,2,0.370104,Asistió,4.0
63431,2026-02-03,CONTROL PRENATAL - MEDICO,1,7,4,0.496083,No asistió,3.0
63433,2026-02-10,JUVENTUD - ODONT,1,8,5,0.541893,No asistió,7.0
63432,2026-02-10,CONTROL PRENATAL - MEDICO,0,8,5,0.541893,No asistió,7.0


In [7]:
# Prueba automática: dias_desde_ultima_cita nunca puede ser menor a 2 (REZAGO_DIAS + 1)
minimo = dataset["dias_desde_ultima_cita"].min()
assert minimo >= params["REZAGO_DIAS"] + 1, "Fuga: el historial está usando información de D-1 o D"
print(f"OK: la cita previa más cercana usada en el historial está a {minimo:.0f} días")

OK: la cita previa más cercana usada en el historial está a 2 días


## 4. Selección de columnas, calidad y exportación

In [8]:
variables_v2 = [v for bloque in fi.BLOQUES_V2.values() for v in bloque]
columnas = fi.COLUMNAS_ID + fi.VARIABLES_V1 + variables_v2
dataset = dataset[columnas].copy()

assert dataset["cita_id"].is_unique
calidad = pd.DataFrame({
    "tipo": dataset.dtypes.astype(str),
    "nulos": dataset.isna().sum(),
    "%_nulos": (dataset.isna().mean() * 100).round(1),
    "valores_unicos": dataset.nunique(),
})
calidad[calidad["nulos"] > 0]

,tipo,nulos,%_nulos,valores_unicos
tasa_inasistencia_hist,float64,71291,21.1,437
dias_desde_ultima_cita,float64,71291,21.1,246
dias_desde_ultima_cita_esp,float64,163079,48.3,249


Los únicos vacíos son esperados: la tasa histórica "cruda" y los días desde la última cita **no existen** si el paciente no tiene historial. Los modelos de árboles los manejan de forma nativa; en la regresión logística se imputan y se agrega un indicador de "sin dato" (dentro del Pipeline).

In [9]:
diccionario = {
    "cita_id": ("Identificación", "Identificador único de la cita"),
    "id_paciente": ("Identificación", "Identificador del paciente (no es variable del modelo)"),
    "fecha_cita": ("Identificación", "Fecha de la cita"),
    "fecha_asig": ("Identificación", "Fecha de asignación"),
    "particion": ("Identificación", "train (ene-may) / validacion (jun) / test (jul) / produccion_simulada (ago-sep)"),
    "no_asiste": ("Objetivo", "1 = no asistió (sin atención en RIPS mismo día y servicio)"),
    "lead_time": ("Agendamiento", "Días entre asignación y cita (>= 1)"),
    "lead_time_cat": ("Agendamiento", "Lead time en rangos"),
    "hora_cita": ("Agendamiento", "Hora de la cita"),
    "franja_horaria": ("Agendamiento", "Franja horaria"),
    "turno_sin": ("Agendamiento", "Hora cíclica (seno)"),
    "turno_cos": ("Agendamiento", "Hora cíclica (coseno)"),
    "es_control": ("Agendamiento", "1 = control, 0 = primera vez"),
    "servicio_homologado": ("Agendamiento", "Servicio real: especialidad + procedimiento agendado (une la codificación vieja y la nueva)"),
    "especialidad": ("Agendamiento", "Especialidad tal como viene en la agenda (no se usa en el modelo: cambió de codificación en abril)"),
    "familia_servicio": ("Agendamiento", "Familia de servicio"),
    "es_pac": ("Agendamiento", "1 = Plan de Atención Complementaria"),
    "edad": ("Sociodemográfica", "Edad en años"),
    "grupo_edad": ("Sociodemográfica", "Curso de vida (Res. 3280/2018)"),
    "sexo": ("Sociodemográfica", "Sexo"),
    "nivel_usr": ("Sociodemográfica", "Nivel del usuario"),
    "tipo_usuario": ("Sociodemográfica", "Cotizante / beneficiario"),
    "regimen": ("Sociodemográfica", "Contributivo / subsidiado"),
    "sede": ("Sociodemográfica", "Sede normalizada"),
    "municipio": ("Sociodemográfica", "Municipio de la sede"),
    "dia_semana_num": ("Calendario", "Día de la semana (1-7)"),
    "dia_semana": ("Calendario", "Día de la semana"),
    "mes": ("Calendario", "Mes"),
    "semana_mes": ("Calendario", "Semana del mes"),
    "es_festivo": ("Calendario", "Día festivo"),
    "es_dia_previo_festivo": ("Calendario", "Día anterior a festivo"),
    "es_dia_posterior_festivo": ("Calendario", "Día posterior a festivo"),
    "es_puente_festivo": ("Calendario", "Puente festivo"),
    "dias_hasta_festivo": ("Calendario", "Días hasta el próximo festivo"),
    "es_primer_dia_habil_mes": ("Calendario", "Primer día hábil del mes"),
    "es_ultimo_dia_habil_mes": ("Calendario", "Último día hábil del mes"),
    "n_citas_previas": ("Historial", "Citas previas (hasta D-2)"),
    "n_asistidas_previas": ("Historial", "Citas previas asistidas"),
    "n_inasistidas_previas": ("Historial", "Citas previas no asistidas"),
    "tasa_inasistencia_hist": ("Historial", "Inasistidas / previas (vacío sin historial)"),
    "tasa_inasistencia_suavizada": ("Historial", "Tasa suavizada hacia el promedio de train"),
    "estado_ultima_cita": ("Historial", "Resultado del último día con citas"),
    "dias_desde_ultima_cita": ("Historial", "Días desde la última cita conocida"),
    "sin_historial": ("Historial", "1 = sin citas previas"),
    "es_cronico": ("Clínica", "Pertenece a programa RCV"),
    "programa_rcv": ("Clínica", "Programa RCV"),
    "diabetes": ("Clínica", "Diabetes según RCV"),
    "gestante_activa": ("Clínica", "Gestante en la fecha de la cita"),
    "n_medicamentos_90d": ("Clínica", "Medicamentos distintos en 90 días previos"),
    "polifarmacia": ("Clínica", "n_medicamentos_90d >= 5"),
    "n_diagnosticos_previos": ("Clínica", "Diagnósticos CIE-10 distintos previos"),
    "n_atenciones_90d": ("Clínica", "Días con atención en 90 días previos"),
    "n_citas_previas_esp": ("Historial v2", "Citas previas en la misma especialidad"),
    "n_inasistidas_previas_esp": ("Historial v2", "Inasistencias previas en la misma especialidad"),
    "tasa_inasistencia_suavizada_esp": ("Historial v2", "Tasa suavizada en la misma especialidad"),
    "estado_ultima_cita_esp": ("Historial v2", "Resultado de la última cita en la especialidad"),
    "dias_desde_ultima_cita_esp": ("Historial v2", "Días desde la última cita en la especialidad"),
    "sin_historial_esp": ("Historial v2", "1 = primera vez en esta especialidad"),
    "n_reprogramaciones_previas": ("Reprogramaciones v2", "Reprogramaciones/cancelaciones previas"),
    "n_otras_citas_mismo_dia": ("Carga agenda v2", "Otras citas del paciente el mismo día"),
    "n_citas_proximos_7d": ("Carga agenda v2", "Citas del paciente en los 7 días siguientes"),
    "n_citas_ultimos_7d": ("Carga agenda v2", "Citas del paciente en los 7 días anteriores"),
    "familia_x_lead": ("Interacciones v2", "Familia de servicio × rango de lead time"),
    "edad_x_control": ("Interacciones v2", "Curso de vida × tipo de cita"),
}
faltantes = set(columnas) - set(diccionario)
assert not faltantes, faltantes

dataset.to_parquet("dataset_modelo.parquet", index=False)
pd.DataFrame([(v, g, d) for v, (g, d) in diccionario.items()], columns=["variable", "grupo", "descripcion"]) \
  .to_csv("diccionario_variables.csv", index=False, encoding="utf-8-sig")

metadatos = {
    "mapa_servicio": {k: int(v) for k, v in mapa_servicio.items()},
    "especialidades_excluidas": list(excluidas),
    "codigos_descontinuados": etiquetadas.attrs.get("descontinuados", []),
    "PRIOR_INASISTENCIA": params["PRIOR_INASISTENCIA"],
    "fecha_construccion": str(pd.Timestamp.today().date()),
}
with open("modelos/metadatos_dataset.json", "w", encoding="utf-8") as archivo:
    json.dump(metadatos, archivo, ensure_ascii=False, indent=2)

print(f"dataset_modelo.parquet: {len(dataset):,} citas, {dataset.shape[1]} columnas, {dataset['id_paciente'].nunique():,} pacientes")

dataset_modelo.parquet: 337,375 citas, 64 columnas, 66,306 pacientes


## 5. Bitácora completa

In [10]:
tabla_bitacora = pd.DataFrame(bitacora)
tabla_bitacora["filas_eliminadas"] = -tabla_bitacora["filas"].diff().fillna(0).astype(int)
tabla_bitacora

,paso,filas,filas_eliminadas
0,Agendamiento original,576252,0
1,Sin duplicados exactos,574730,1522
2,"Con fecha de asignación, especialidad y hora",567212,7518
3,Una cita por paciente-día-servicio,516498,50714
4,Dentro de la ventana con RIPS,483541,32957
5,Asignadas al menos 1 día antes (lead time >= 1),442516,41025
6,Solo consultas (sin procedimientos),399008,43508
7,Sin terapias ni especialidades no reportadas,381955,17053
8,Sin citas huérfanas de códigos descontinuados,378598,3357
9,Sin días sin reporte en RIPS,359677,18921
